<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · Answers you can trust →](02_Answers_You_Can_Trust.ipynb)

# 1 · Constraints as data

**The problem.** A shipment of vaccine must stay between 2 and 8 °C. That constraint lives in the QA team's spreadsheet,
in the Python pipeline that processes the data loggers' uploads, in the TypeScript dashboard that flags shipments, and
in the logger's firmware, which sounds an alarm. Four copies, in four languages. When the range changes for a new
product, someone updates three of them.

A constraint in code can't be stored, sent or read by another language: a Python function means nothing to the
dashboard. mbse-expressions writes the constraint once, as an **expression**: data with a schema, like any mbse-schemas
object, that every program loads and evaluates the same way, and that translates into each of those languages (case
study 7).

This case study writes a constraint, evaluates it, checks it, and saves it. The cold chain's schemas are in
`toolkit.py`, beside this notebook: a `Shipment` has the latest temperature its logger reported (`celsius`), every
reading (`readings`), the hours it has been in transit (`hours`), and the loggers it carries.

In [1]:
import json

from mbse.Expressions import Evaluators, Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Terms
from mbse.Schemas.Framework import JSON
from toolkit import cold_chain, show

store = cold_chain()
s1 = store.Shipment().id("S1").celsius(5.5).hours(30).create()
s2 = store.Shipment().id("S2").celsius(9.1).hours(80).create()

## Step 1: writing a constraint

In Python, the easiest way to write a constraint is as a lambda. `Text.FromFunction` reads the lambda's **source**,
never calling it, and builds the expression it describes. Its parameter, `this`, becomes a variable: the shipment the
constraint is about.

In [2]:
in_range = Text.FromFunction(lambda this: 2.0 <= this.celsius <= 8.0)
print(show(in_range))

and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0))


`show` (from the toolkit) writes the expression as function calls. `this.celsius` reads a property: `get`. The
chained comparison is two comparisons, `le`, joined by `and`. These operation names are the **core vocabulary** that
every program evaluating expressions implements, in Python and in TypeScript alike.

## Step 2: evaluating it

`Evaluators.OfAny(constraint, scope)` computes a constraint's value, with the variables in the scope bound to values:

In [3]:
Evaluators.OfAny(in_range, {"this": s1}), Evaluators.OfAny(in_range, {"this": s2})

(True, False)

A constraint about one value calls it `this`, so checking it against each value is evaluating it with `this` bound:

In [4]:
[Evaluators.OfAny(in_range, {"this": shipment}) for shipment in (s1, s2)]

[True, False]

## Step 3: writers build the same data

A lambda is a convenience of Python's. Underneath, and in TypeScript, constraints are built with **writers**: a
variable's writer reads properties as attributes and builds operations with methods. The two notations build the same
data, as `Terms.same` (structural equality) confirms:

In [5]:
this = E.variable("this")
by_hand = this.celsius.ge(2.0).and_(this.celsius.le(8.0))
print(show(by_hand))
Terms.same(by_hand.data, Text.FromFunction(lambda this: this.celsius >= 2.0 and this.celsius <= 8.0).data)

and(ge(get(this, 'celsius'), 2.0), le(get(this, 'celsius'), 8.0))


True

`ge` is `this.celsius >= 2.0`, the same comparison as `2.0 <= this.celsius`, written the other way round: not the
same expression, which is why the second constraint is spelled that way. Writers are what a program uses when it
assembles constraints from parts, as a constraint editor would.

## Step 4: what a constraint can say

A constraint can have several variables, and read names from your program. A name that isn't a parameter, like
`MAX_HOURS`, is read **once**, when `FromFunction` runs, and becomes a literal: the expression doesn't refer back to
your program, so it means the same thing wherever it's loaded.

In [6]:
MAX_HOURS = 72
on_time = Text.FromFunction(lambda this: this.hours <= MAX_HOURS)
late_by = Text.FromFunction(lambda this, now: this.hours + now > MAX_HOURS)
print(show(on_time))
print(show(late_by))
Evaluators.OfAny(late_by, {"this": s1, "now": 50})

le(get(this, 'hours'), 72)
gt(add(get(this, 'hours'), now), 72)


True

To name an intermediate value, call a one-parameter lambda. That's a **let**: the name is bound to the value inside
the body, and nowhere else.

In [7]:
print(show(Text.FromFunction(lambda this: (lambda c: c >= 2.0 and c <= 8.0)(this.celsius))))

let c = get(this, 'celsius') in and(ge(c, 2.0), le(c, 8.0))


Only what maps onto the core vocabulary is accepted, and the rest is refused when you write the constraint, not when it
is evaluated. The core has no division, because division by zero would make a constraint partial:

In [8]:
try:
    Text.FromFunction(lambda this: this.celsius / 2 > 4.0)
except ValueError as error:
    print(error)

cannot convert 'this.celsius / 2': not supported in an expression


## Step 5: validating a constraint before it is evaluated

A constraint that arrives as data, from a file or another program, may be broken. `validate()` reports what evaluating
it would raise: variables nothing binds, wrong numbers of arguments, and, with `core=True`, operations outside the core
vocabulary, which some programs don't implement.

In [9]:
print(late_by.data.validate(bound={"this", "now"}, core=True))
print(late_by.data.validate(bound={"this"}))
print(E.operation("div", this.celsius, 2).data.validate(bound={"this"}, core=True))

[]
["argument 0: argument 1: variable 'now' is not bound"]
["'div' is not a core operation"]


`.data` is the expression a writer holds; most functions accept either.

## Step 6: saving and loading

An expression's data has a schema, so it saves like any object graph. `E.DIALECT.schema_of(constraint)` gives the schema
to save it by, and `E.Builders` builds expressions back from the text:

In [10]:
schema = E.DIALECT.schema_of(in_range.data)
text = JSON.ToJSON(E.Builders).Reachable(schema, in_range.data)
print(json.dumps(json.loads(text), indent=1))  # indented, to read

{
 "root": "s0",
 "objects": {
  "s0": {
   "kind": "operation",
   "name": "and",
   "arguments": [
    {
     "argument": {
      "$ref": "s1",
      "$schema": "Expressions.OfOperation"
     },
     "index": 0
    },
    {
     "argument": {
      "$ref": "s2",
      "$schema": "Expressions.OfOperation"
     },
     "index": 1
    }
   ]
  },
  "s1": {
   "kind": "operation",
   "name": "le",
   "arguments": [
    {
     "argument": {
      "$ref": "s3",
      "$schema": "Expressions.OfLiteral"
     },
     "index": 0
    },
    {
     "argument": {
      "$ref": "s4",
      "$schema": "Expressions.OfOperation"
     },
     "index": 1
    }
   ]
  },
  "s2": {
   "kind": "operation",
   "name": "le",
   "arguments": [
    {
     "argument": {
      "$ref": "s4",
      "$schema": "Expressions.OfOperation"
     },
     "index": 0
    },
    {
     "argument": {
      "$ref": "s5",
      "$schema": "Expressions.OfLiteral"
     },
     "index": 1
    }
   ]
  },
  "s3": {
   "kind": "li

Each object carries a `kind`, arguments are entries with an `index`, and `get(this, 'celsius')`, which the constraint
uses twice, is written once. Nothing in the text is Python: the TypeScript dashboard reads the same JSON, byte for byte,
and evaluates the same constraint.

In [11]:
loaded = JSON.FromJSON(E.Builders).Reachable(schema, text)
print(show(loaded), Terms.same(loaded, in_range.data))
[Evaluators.OfAny(loaded, {"this": shipment}) for shipment in (s1, s2)]

and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)) True


[True, False]

## Why it works this way

- **A constraint is data, not code.** Data can be stored beside what it's about, versioned, diffed, sent, and loaded by
  a program in another language. Code can only be run, and only by its own language.
- **A small core vocabulary.** Every operation in the core is pure, total and bounded, so every implementation can
  evaluate it the same way, and a solver could reason about it. What isn't in the core is an extension, reported by
  `validate(core=True)`.
- **Names are read once.** A constraint that referred back into the program that wrote it couldn't be loaded anywhere
  else.

The next case study looks at what the constraints answer when the data is incomplete, which is most of the time.

<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [2 · Answers you can trust →](02_Answers_You_Can_Trust.ipynb)